# 🔬 Caderno 04: Abordagem V2 - Razões Elementares, Padronização Interna e Interpretabilidade
**Projeto:** Detecção de Adulteração em Fertilizantes NPK via LIBS  
**Instituição:** Embrapa Instrumentação  

---

### Contexto da Abordagem V2:
Modelos do espectro completo (como PLSR e RF) são eficazes, mas funcionam como "caixas-pretas" de alta dimensionalidade.  
Na **Abordagem V2**, aplicamos **padronização interna estequiométrica**:
1. **Espectro Representativo Médio** por pastilha para eliminar disparos individuais ruidosos e aumentar a razão sinal-ruído.
2. **Integração Numérica de Áreas de Pico** para linhas de emissão específicas dos elementos adulterantes e da matriz:
   - **Canal VIS:** Razões Cálcio/Potássio (**Ca/K**), ex: `Ca_527/K_693`, `Ca_315/K_580`.
   - **Canal UV:** Razões Cálcio/Fósforo (**Ca/P**) e Magnésio/Fósforo (**Mg/P**).
3. **Métrica de Distância Interclasse de Cohen ($D_{C-K}$):** Ranqueamento automático das razões que melhor separam o fertilizante genuíno dos adulterados.
4. **Modelos Parcimoniosos Lineares (OLS)** com interpretação físico-química direta da inclinação (coeficiente angular).

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from src.features import (
    gerar_espectro_representativo,
    extrair_areas_picos,
    calcular_razoes_elementares,
    calcular_distancia_interclasse_cohen,
    PICOS_INTERESSE_VIS,
    PICOS_INTERESSE_UV
)
from src.models import (
    regressao_linear_razoes
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix

print("Módulos carregados com sucesso!")

## 1. Carregamento dos Dados e Geração dos Espectros Representativos
Calculamos o espectro médio de cada pastilha para eliminar o efeito de disparos individuais.

In [ ]:
pasta_proc = BASE_DIR / "data" / "processed"
caminho_vis = pasta_proc / "df_limpo_vis.parquet"

df_vis = pd.read_parquet(caminho_vis) if caminho_vis.exists() else None

if df_vis is not None:
    # Remove amostras de teste ou descartadas
    df_vis_filtrado = df_vis[df_vis['Amostra_ID'] != 'i_'].copy()

    # Gera espectro representativo por pastilha
    df_rep_vis = gerar_espectro_representativo(df_vis_filtrado)
    print(f"Disparos individuais originais: {len(df_vis_filtrado)}")
    print(f"Pastilhas representativas independentes: {len(df_rep_vis)}")
    display(df_rep_vis[['Teor_Adulteracao', 'Amostra_ID', 'Adulterado']]) if 'display' in dir() else print(df_rep_vis[['Teor_Adulteracao', 'Amostra_ID']].head())
else:
    print("Aviso: execute primeiro o Caderno 01 para gerar os dados tratados.")

## 2. Integração Numérica das Áreas de Pico (VIS)
Integramos as linhas de emissão do Potássio (K a 693.96, 404.46, 580.20 nm) e Cálcio (Ca a 527.08, 315.96 nm).

In [ ]:
if df_vis is not None:
    df_areas_vis = extrair_areas_picos(df_rep_vis, PICOS_INTERESSE_VIS, janela_nm=0.3)
    print("Áreas integradas para cada pico:")
    display(df_areas_vis.head()) if 'display' in dir() else print(df_areas_vis.head())

## 3. Cálculo das Razões Elementares e Ranqueamento via Distância de Cohen ($D_{C-K}$)
Calculamos todas as combinações Ca/K e avaliamos qual razão maximiza a distância padronizada entre a amostra pura e as adulteradas.

In [ ]:
if df_vis is not None:
    picos_ca = ['Ca_527', 'Ca_315']
    picos_k = ['K_693', 'K_404', 'K_580']

    df_razoes_vis, colunas_razao = calcular_razoes_elementares(
        df_areas_vis, numeradores=picos_ca, denominadores=picos_k
    )

    # Ranking da distância interclasse de Cohen (D_C-K)
    df_tabela_dck = calcular_distancia_interclasse_cohen(df_razoes_vis, colunas_razao, teor_referencia=0.0)
    print("=== Distância Interclasse (D_C-K) por Razão ===")
    display(df_tabela_dck) if 'display' in dir() else print(df_tabela_dck.to_string())

    melhor_razao_1 = df_tabela_dck.iloc[0]['Razão Analisada']
    melhor_razao_2 = df_tabela_dck.iloc[1]['Razão Analisada']
    print(f"\nAs duas melhores razões selecionadas: {melhor_razao_1} e {melhor_razao_2}")

## 4. Visualização 2D da Separação por Razões Elementares
Dispersão das amostras no espaço bidimensional das melhores razões de Ca/K.

In [ ]:
if df_vis is not None:
    df_razoes_vis['Teor_Str'] = df_razoes_vis['Teor_Adulteracao'].apply(lambda x: f"{float(x):.1f}%")

    plt.figure(figsize=(9, 6))
    sns.scatterplot(
        data=df_razoes_vis,
        x=melhor_razao_1,
        y=melhor_razao_2,
        hue='Teor_Str',
        palette='Set1',
        s=130,
        edgecolor='black'
    )

    for i in range(len(df_razoes_vis)):
        plt.text(
            df_razoes_vis[melhor_razao_1].iloc[i],
            df_razoes_vis[melhor_razao_2].iloc[i],
            df_razoes_vis['Amostra_ID'].iloc[i],
            fontsize=9, ha='left', va='bottom'
        )

    plt.title(f"Separação 2D de Adulteração: Razões {melhor_razao_1} vs {melhor_razao_2}", fontsize=13, fontweight='bold')
    plt.xlabel(f"Razão Ca/K ({melhor_razao_1})", fontsize=11)
    plt.ylabel(f"Razão Ca/K ({melhor_razao_2})", fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend(title="Teor Real", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## 5. Modelagem de Classificação com KNN nas Razões 2D (LOO-CV)

In [ ]:
if df_vis is not None:
    X_2d = df_razoes_vis[[melhor_razao_1, melhor_razao_2]].values
    y_2d = df_razoes_vis['Teor_Str'].values

    loo = LeaveOneOut()
    knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean')
    y_pred_knn = cross_val_predict(knn, X_2d, y_2d, cv=loo)

    acuracia = accuracy_score(y_2d, y_pred_knn)
    print(f"Acurácia do KNN 2D (Leave-One-Out CV): {acuracia*100:.2f}%")

    labels_unicos = sorted(list(np.unique(y_2d)))
    cm = confusion_matrix(y_2d, y_pred_knn, labels=labels_unicos)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=labels_unicos, yticklabels=labels_unicos)
    plt.title(f"Matriz de Confusão 2D (Acurácia: {acuracia*100:.1f}%)")
    plt.xlabel("Predição do KNN")
    plt.ylabel("Real")
    plt.show()

## 6. Regressão Linear Parcimoniosa (OLS) e Interpretabilidade
Avaliamos a relação linear direta entre a razão Ca/K e a concentração de adulteração com Leave-One-Out CV.

In [ ]:
if df_vis is not None:
    res_linear = regressao_linear_razoes(
        df_razoes_vis,
        features=[melhor_razao_1],
        col_alvo='Teor_Adulteracao'
    )

    print("--- Regressão Linear Univariada (OLS) ---")
    print(f"Equação: Teor (%) = {res_linear['coeficientes'][0]:.4f} * ({melhor_razao_1}) + {res_linear['intercept']:.4f}")
    print(f"R² (LOO-CV): {res_linear['R2_CV']:.4f}")
    print(f"RMSE (LOO-CV): {res_linear['RMSE_CV']:.2f}% de teor")

    # Gráfico de dispersão com reta ajustada
    plt.figure(figsize=(8, 5))
    sns.regplot(
        data=df_razoes_vis,
        x=melhor_razao_1,
        y='Teor_Adulteracao',
        scatter_kws={'s': 80, 'color': 'tab:blue', 'edgecolor': 'black'},
        line_kws={'color': 'tab:red', 'label': f"Reta Ajustada (R² = {res_linear['R2_CV']:.3f})"}
    )
    plt.title(f"Curva Analítica de Calibração: {melhor_razao_1} vs Teor de Adulteração", fontsize=13, fontweight='bold')
    plt.xlabel(f"Razão Espectral ({melhor_razao_1})", fontsize=11)
    plt.ylabel("Teor Real de Adulteração (%)", fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.legend()
    plt.show()

## 7. Regressão Linear Multivariada (Duas Razões)

In [ ]:
if df_vis is not None:
    res_multi = regressao_linear_razoes(
        df_razoes_vis,
        features=[melhor_razao_1, melhor_razao_2],
        col_alvo='Teor_Adulteracao'
    )

    print("--- Regressão Linear Bivariada ---")
    print(f"R² (LOO-CV): {res_multi['R2_CV']:.4f}")
    print(f"RMSE (LOO-CV): {res_multi['RMSE_CV']:.2f}% de teor")
    print(f"Coeficientes: {res_multi['coeficientes']}")
    print(f"Intercept: {res_multi['intercept']:.4f}")